In [0]:
WITH ranked AS (
  SELECT
    g.market,
    p.category_name,
    ROUND(SUM(f.profit), 0) AS total_profit,
    RANK() OVER (PARTITION BY g.market ORDER BY SUM(f.profit) DESC) AS profit_rank
  FROM workspace.gold.fct_order_items AS f
  INNER JOIN workspace.gold.dim_geography AS g ON f.geography_key = g.geography_key
  INNER JOIN workspace.gold.dim_product AS p ON f.product_key = p.product_key
  GROUP BY g.market, p.category_name
)
SELECT * FROM ranked
WHERE profit_rank <= 3
ORDER BY market, profit_rank

In [0]:
WITH monthly AS (
  SELECT d.year_month, SUM(f.profit) AS profit
  FROM workspace.gold.fct_order_items AS f
  INNER JOIN workspace.gold.dim_date AS d ON f.date_key = d.date_key
  GROUP BY d.year_month
)
SELECT
  year_month,
  ROUND(profit, 0) AS profit,
  ROUND(profit - LAG(profit) OVER (ORDER BY year_month), 0) AS mom_change,
  ROUND(100 * (profit / NULLIF(LAG(profit) OVER (ORDER BY year_month), 0) - 1), 1) AS mom_pct,
  ROUND(AVG(profit) OVER (ORDER BY year_month ROWS BETWEEN 2 PRECEDING AND CURRENT ROW), 0) AS profit_3m_avg
FROM monthly
ORDER BY year_month

In [0]:
SELECT
  sm.shipping_mode,
  COUNT(*) AS items,
  ROUND(100 * AVG(CASE WHEN f.is_late THEN 1 ELSE 0 END), 1) AS late_pct,
  percentile_cont(0.5) WITHIN GROUP (ORDER BY f.shipping_delay_days) AS median_delay_days,
  percentile_cont(0.9) WITHIN GROUP (ORDER BY f.shipping_delay_days) AS p90_delay_days
FROM workspace.gold.fct_order_items AS f
INNER JOIN workspace.gold.dim_shipping_mode AS sm ON f.shipping_mode_key = sm.shipping_mode_key
GROUP BY sm.shipping_mode
ORDER BY late_pct DESC

In [0]:
WITH bucketed AS (
  SELECT
    CASE
      WHEN discount_rate < 0.05 THEN '1) under 5%'
      WHEN discount_rate < 0.15 THEN '2) 5% to 15%'
      ELSE '3) 15% or more'
    END AS discount_band,
    profit
  FROM workspace.gold.fct_order_items
)
SELECT
  discount_band,
  COUNT(*) AS items,
  SUM(CASE WHEN profit < 0 THEN 1 ELSE 0 END) AS loss_items,
  ROUND(100 * AVG(CASE WHEN profit < 0 THEN 1 ELSE 0 END), 1) AS loss_pct,
  ROUND(SUM(profit), 0) AS total_profit
FROM bucketed
GROUP BY discount_band
ORDER BY discount_band

In [0]:
SELECT 'duplicate order items' AS check_name,
       COUNT(*) - COUNT(DISTINCT order_item_key) AS issue_rows
FROM workspace.gold.fct_order_items
UNION ALL
SELECT 'negative shipping days', COUNT(*)
FROM workspace.gold.fct_order_items WHERE shipping_days_real < 0
UNION ALL
SELECT 'shipped before ordered', COUNT(*)
FROM workspace.gold.fct_order_items WHERE shipping_date_key < date_key
UNION ALL
SELECT 'quantity zero or negative', COUNT(*)
FROM workspace.gold.fct_order_items WHERE quantity <= 0
UNION ALL
SELECT 'customers with no orders', COUNT(*)
FROM workspace.gold.dim_customer AS c
LEFT JOIN workspace.gold.fct_order_items AS f ON c.customer_key = f.customer_key
WHERE f.customer_key IS NULL